# Interactuando con Python, Pandas y Cloud SQL - PostgreSQL

Instalamos lo necesario

In [1]:
!pip install psycopg2-binary
!pip install pandas
!pip install sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 11.3 MB/s eta 0:00:00


Permitimos a Colab que interactue con GCP

In [ ]:
from google.colab import auth
auth.authenticate_user()

Importamos librerías

In [4]:
import pandas as pd
import json
import requests

Tomaremos como ejemplo los archivos schema.json y part-00000

Definimos el nombre de las columnas en nuestro archivo

In [31]:
schemas = json.load(open('./schemas.json'))
columns = [col['column_name'] for col in sorted(schemas['orders'], key=lambda col: col['column_position'])]
columns

['order_id', 'order_date', 'order_customer_id', 'order_status']

Vista previa de nuestro archivo

In [12]:
orders = pd.read_csv('./part-00000', names=columns)
orders

Query/transformación de los datos

In [16]:
daily_status_count = orders. \
    groupby(['order_date', 'order_status'])['order_id']. \
    agg(order_count='count'). \
    reset_index()
daily_status_count

Definimos los parámetros de conexión a nuestra base de datos

In [20]:
host = '35.193.226.238'
port = 5432
database = 'test'
user = 'postgres'
password = 'Passw0rd'

In [21]:
conn_uri = f'postgresql://{user}:{password}@{host}:{port}/{database}'

Para que Colab pueda conectarse, necesitamos añadir su IP en la consola o con gcloud:
```gcloud sql instances patch sd-local-sql-pg-1 --authorized-networks <ip-addres>```

In [23]:
ip = requests.get('https://api.ipify.org').text
print(f'Current IP Address: {ip}')

Current IP Address: 35.196.243.80


Una vez configurado, insertamos la data en nuestra base de datos.

In [24]:
daily_status_count.to_sql(
    'daily_status_count',
    conn_uri,
    if_exists='replace',
    index=False
)

203

La leemos

In [25]:
dsc = pd.read_sql(
    'daily_status_count',
    conn_uri
)
dsc

Probamos otro query

In [28]:
dsc = pd.read_sql(
    '''
        SELECT order_status, sum(order_count) AS order_count FROM daily_status_count
        GROUP BY 1
        ORDER BY 2 DESC
    ''',
    conn_uri
)
dsc